# Lab 2: ทำความรู้จัก Embeddings

## เป้าหมาย

1. สร้าง Dense Embedding ด้วย LangChain
2. อธิบายความแตกต่างระหว่าง **Sparse Embedding** และ **Dense Embedding**
3. คำนวณ **Cosine Similarity** ระหว่างข้อความสองข้อความ
4. คำพ้อง การสะกดผิด และการแปลข้ามภาษา
5. (Optional) ลองใช้ FAISS และ Qdrant vector database

In [2]:
!uv pip install -q langgraph langchain-google-genai numpy pandas scipy scikit-learn matplotlib seaborn

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")

# Embedding คืออะไร?

**Embedding** คือ การแทนข้อมูล เช่น คำ ประโยค หรือเอกสาร ด้วยเวกเตอร์ของตัวเลข 

**Sparse Embedding**

- (มักจะ)แทน ประโยค ด้วย ความถี่ของคำ เช่น Bag-of-Words และ TF–IDF
- มีจำนวนมิติมาก แต่ค่าจำนวนมากเป็นศูนย์

**Dense Embedding**

- สร้างจากโมเดลที่เรียนรู้รูปแบบภาษาและบริบท
- มีจำนวนมิติไม่มาก และเกือบทุกมิติมีค่า
- เหมาะกับ Semantic Search เพราะจับความใกล้เคียงเชิงความหมายได้ดีกว่า


**Embedding Dimensions**

| Provider/model                    | Native or default dimension |                             Configurable dimensions | Raw storage per 1M vectors¹ |
| --------------------------------- | --------------------------: | --------------------------------------------------: | --------------------------: |
| **Gemini Embedding 2**            |                       3,072 | Up to 3,072; Google recommends 768, 1,536, or 3,072 |                    12.29 GB |
| **Gemini Embedding 001**          |                       3,072 |          Up to 3,072; commonly 768, 1,536, or 3,072 |                    12.29 GB |
| **Qwen3-Embedding-0.6B**          |                       1,024 |                                            32–1,024 |                     4.10 GB |
| **Qwen3-Embedding-4B**            |                       2,560 |                          Can be truncated using MRL |                    10.24 GB |
| **Qwen3-Embedding-8B**            |                       4,096 |                          Can be truncated using MRL |                    16.38 GB |
| **OpenAI text-embedding-3-small** |                       1,536 |                                         Up to 1,536 |                     6.14 GB |
| **OpenAI text-embedding-3-large** |                       3,072 |                                         Up to 3,072 |                    12.29 GB |

### เปรียบเทียบ Dense vs Sparse Embeddings

In [2]:
sentences = [
    "แมวของฉันชอบนอนกลางแดด",
    "เจ้าเหมียวที่บ้านชอบไปนอนตากแดด",
    "ลูกแมวกำลังวิ่งไล่จับลูกบอล",
    "เจ้าเหมียวเล่นของเล่นอย่างสนุกสนาน",
    "แมวของฉันชอบเล่นกับกล่องกระดาษ",
    "แมวตัวนี้ชอบกินปลาทูน่า",
    "เจ้าเหมียวกำลังกินอาหารอยู่ในครัว",
    "ลูกแมวดื่มนมจากชามใบเล็ก",
    "แมวส่งเสียงร้องเมื่อต้องการอาหาร",
    "เจ้าเหมียวร้องเหมียวเพราะกำลังหิว",
    "แมวสีดำกำลังปีนขึ้นไปบนต้นไม้",
    "แมวกระโดดขึ้นไปนั่งบนกำแพง",
    "แมวของฉันกลัวเสียงเครื่องดูดฝุ่น",
    "เจ้าเหมียวรีบวิ่งหนีเมื่อได้ยินเสียงดัง",
    "แมวใช้ลิ้นเลียขนเพื่อทำความสะอาดตัวเอง",
    "เจ้าเหมียวกำลังแต่งขนอย่างสบายใจ",
    "สัตวแพทย์กำลังตรวจสุขภาพของลูกแมว",
    "แมวควรได้รับวัคซีนและตรวจสุขภาพเป็นประจำ",
    "แมวเป็นสัตว์เลี้ยงที่รักอิสระและชอบสำรวจสิ่งรอบตัว",
    "วันนี้ฝนตกหนักมากที่กรุงเทพ",
]

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity


In [4]:
def build_sparse_embeddings(texts):
    vectorizer = CountVectorizer(analyzer="char_wb", ngram_range=(3, 5), lowercase=True,)
    matrix = vectorizer.fit_transform(texts)

    return vectorizer, texts, matrix

def get_sparse_embeddings(vectorizer, texts):
    matrix = vectorizer.transform(texts)

    return matrix

In [5]:
vectorizer, texts, sparse_vectors = build_sparse_embeddings(sentences)

In [6]:
vectorizer

,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(3, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'char_wb'
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'


In [7]:
print("Embedding size:", len(vectorizer.get_feature_names_out()))

Embedding size: 1443


In [8]:
vectorizer.get_feature_names_out()[0:100]

array([' ลู', ' ลูก', ' ลูกแ', ' วั', ' วัน', ' วันน', ' สั', ' สัต',
       ' สัตว', ' เจ', ' เจ้', ' เจ้า', ' แม', ' แมว', ' แมวก', ' แมวข',
       ' แมวค', ' แมวต', ' แมวส', ' แมวเ', ' แมวใ', 'กชา', 'กชาม',
       'กชามใ', 'กที', 'กที่', 'กที่ก', 'กบอ', 'กบอล', 'กบอล ', 'กมา',
       'กมาก', 'กมากท', 'กระ', 'กระด', 'กระดา', 'กระโ', 'กระโด', 'กรุ',
       'กรุง', 'กรุงเ', 'กลั', 'กลัว', 'กลัวเ', 'กลา', 'กลาง', 'กลางแ',
       'กล่', 'กล่อ', 'กล่อง', 'กสน', 'กสนา', 'กสนาน', 'กหน', 'กหนั',
       'กหนัก', 'กอิ', 'กอิส', 'กอิสร', 'กับ', 'กับก', 'กับกล', 'การ',
       'การอ', 'การอา', 'กำล', 'กำลั', 'กำลัง', 'กำแ', 'กำแพ', 'กำแพง',
       'กิน', 'กินป', 'กินปล', 'กินอ', 'กินอา', 'กแด', 'กแดด', 'กแดด ',
       'กแม', 'กแมว', 'กแมว ', 'กแมวก', 'กแมวด', 'ขนอ', 'ขนอย', 'ขนอย่',
       'ขนเ', 'ขนเพ', 'ขนเพื', 'ขภา', 'ขภาพ', 'ขภาพข', 'ขภาพเ', 'ของ',
       'ของฉ', 'ของฉั', 'ของล', 'ของลู', 'ของเ'], dtype=object)

In [9]:
for text, vec in zip(texts[0:3], sparse_vectors[0:3]):
    print("text:", text)
    print("vector:", vec[:, 0:10].toarray())
    print("len(vector):", vec.shape)
    print()

text: แมวของฉันชอบนอนกลางแดด
vector: [[0 0 0 0 0 0 0 0 0 0]]
len(vector): (1, 1443)

text: เจ้าเหมียวที่บ้านชอบไปนอนตากแดด
vector: [[0 0 0 0 0 0 0 0 0 1]]
len(vector): (1, 1443)

text: ลูกแมวกำลังวิ่งไล่จับลูกบอล
vector: [[1 1 1 0 0 0 0 0 0 0]]
len(vector): (1, 1443)



In [10]:
MODEL_NAME = "gemini-embedding-2"
OUTPUT_DIMENSION = 768

embedder = GoogleGenerativeAIEmbeddings(
    model=MODEL_NAME,
    task_type="SEMANTIC_SIMILARITY",
    output_dimensionality=OUTPUT_DIMENSION,
)

def generate_dense_embeddings(texts):
    vectors = embedder.embed_documents(texts)
    return texts, vectors


In [11]:
texts, dense_vectors = generate_dense_embeddings(sentences[0:3])

In [12]:
for text, vec in zip(texts, dense_vectors):
    print("text:", text)
    print("vector:", vec[0:5])
    print("len(vector):", len(vec))
    print()

text: แมวของฉันชอบนอนกลางแดด
vector: [-0.023844179, -0.009347884, -0.004376163, 0.01714996, -0.03071677]
len(vector): 768

text: เจ้าเหมียวที่บ้านชอบไปนอนตากแดด
vector: [-0.009118847, -0.019067068, -0.0074945767, -0.015018896, -0.016160337]
len(vector): 768

text: ลูกแมวกำลังวิ่งไล่จับลูกบอล
vector: [-0.0058138603, -0.015632655, 0.0012679762, 0.027369162, 0.01718484]
len(vector): 768



In [14]:
import pandas as pd
import numpy as np
vector_summary = pd.DataFrame([
    {
        "ชนิด": "Dense",
        "จำนวนมิติ": len(dense_vectors[0]),
        "จำนวนค่าที่ไม่เป็นศูนย์": int(np.count_nonzero(dense_vectors[0])),
        "สัดส่วนค่าที่ไม่เป็นศูนย์": np.count_nonzero(dense_vectors[0]) / len(dense_vectors[0]),
    },
    {
        "ชนิด": "Sparse (TF)",
        "จำนวนมิติ": sparse_vectors.shape[1],
        "จำนวนค่าที่ไม่เป็นศูนย์": int(sparse_vectors[0, :].nnz),
        "สัดส่วนค่าที่ไม่เป็นศูนย์": sparse_vectors[0, :].nnz / sparse_vectors.shape[1],
    },
])

display(vector_summary.style.format({"สัดส่วนค่าที่ไม่เป็นศูนย์": "{:.2%}"}))


,ชนิด,จำนวนมิติ,จำนวนค่าที่ไม่เป็นศูนย์,สัดส่วนค่าที่ไม่เป็นศูนย์
0,Dense,768,768,100.00%
1,Sparse (TF),1443,63,4.37%


# Cosine Similarity

วิธีในการเปรียบเทียบเวกเตอร์ $\mathbf{a}$ และ $\mathbf{b}$

$$
\operatorname{cosine\_similarity}(\mathbf{a},\mathbf{b})
= \frac{\mathbf{a}\cdot\mathbf{b}}
{\lVert\mathbf{a}\rVert\,\lVert\mathbf{b}\rVert}
$$

โดยทั่วไป ค่ายิ่งสูง แสดงว่า ทิศทางของเวกเตอร์ยิ่งใกล้กัน = $\mathbf{a}$ และ $\mathbf{b}$ มีแนวโน้มว่า จะมีเนื้อหาคล้ายๆกัน


In [15]:
import numpy as np

def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    '''คำนวณ Cosine Similarity ระหว่าง Dense Vector สองเวกเตอร์'''
    a = np.asarray(vec_a, dtype=np.float64)
    b = np.asarray(vec_b, dtype=np.float64)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)


# ตรวจสอบฟังก์ชันด้วยเวกเตอร์ขนาดเล็ก
print("ทิศทางเดียวกัน:", cosine_similarity([1, 0], [2, 0]))
print("ตั้งฉากกัน:", cosine_similarity([1, 0], [0, 1]))


ทิศทางเดียวกัน: 1.0
ตั้งฉากกัน: 0.0


In [16]:
tests = [
    ("Spelling variations", ["แมวชอบนอนกลางวัน", "แมวชอบนอนกลงวัน", "แมวชอบบบบบนอนกลางวัน", "ประเทศไทยร้อนมาก"]),
    ("Synonyms", ["แมวกำลังวิ่งอย่างรวดเร็ว", "เจ้าเหมียวเคลื่อนที่อย่างว่องไว", "ประเทศไทยร้อนมาก"]),
    ("Translations", ["ลูกแมวกำลังเล่นลูกบอล", "A kitten is playing with a ball."]),
]

In [17]:
from itertools import combinations
import pandas as pd
from tqdm import tqdm

results = []

for test_name, texts in tqdm(tests):
    _, vectors = generate_dense_embeddings(texts)

    for i, j in combinations(range(len(texts)), 2):
        score = cosine_similarity(vectors[i], vectors[j])

        results.append({
            "test": test_name,
            "text_a": texts[i],
            "text_b": texts[j],
            "cosine_similarity": score,
        })

results_df = pd.DataFrame(results)
results_df["cosine_similarity"] = results_df["cosine_similarity"].round(4)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.85it/s]


In [18]:
for test_name in results_df["test"].unique():
    print(f"\n{'=' * 40}")
    print(f"Test: {test_name}")
    print("=" * 40)

    display(
        results_df.loc[
            results_df["test"] == test_name,
            ["text_a", "text_b", "cosine_similarity"],
        ]
    )


Test: Spelling variations


,text_a,text_b,cosine_similarity
0,แมวชอบนอนกลางวัน,แมวชอบนอนกลงวัน,0.8714
1,แมวชอบนอนกลางวัน,แมวชอบบบบบนอนกลางวัน,0.8999
2,แมวชอบนอนกลางวัน,ประเทศไทยร้อนมาก,0.4895
3,แมวชอบนอนกลงวัน,แมวชอบบบบบนอนกลางวัน,0.8396
4,แมวชอบนอนกลงวัน,ประเทศไทยร้อนมาก,0.4741
5,แมวชอบบบบบนอนกลางวัน,ประเทศไทยร้อนมาก,0.5282



Test: Synonyms


,text_a,text_b,cosine_similarity
6,แมวกำลังวิ่งอย่างรวดเร็ว,เจ้าเหมียวเคลื่อนที่อย่างว่องไว,0.6739
7,แมวกำลังวิ่งอย่างรวดเร็ว,ประเทศไทยร้อนมาก,0.4382
8,เจ้าเหมียวเคลื่อนที่อย่างว่องไว,ประเทศไทยร้อนมาก,0.3575



Test: Translations


,text_a,text_b,cosine_similarity
9,ลูกแมวกำลังเล่นลูกบอล,A kitten is playing with a ball.,0.7593


# Matryoshka Representation

**Matryoshka Representation Learning (MRL)** คือ เทคนิคการเทรน เพื่อให้ข้อมูลสำคัญอยู่ในมิติต้น ๆ ของ embedding คล้ายตุ๊กตาแม่ลูกดก (Matryoshka) ที่เวกเตอร์ขนาดเล็กซ้อนอยู่ในเวกเตอร์ขนาดใหญ่ ดังนั้นเราสามารถเก็บเฉพาะ prefix เช่น 128 หรือ 256 มิติจากเวกเตอร์ 768 มิติ เพื่อลดพื้นที่จัดเก็บและเวลาในการค้นหา โดยคุณภาพมักลดลงเพียงเล็กน้อย

> หลังตัดมิติควร normalize เวกเตอร์ใหม่ก่อนคำนวณ cosine similarity และควรทำวิธีนี้เฉพาะกับโมเดลที่รองรับ MRL เท่านั้น เพราะ embedding ทั่วไปไม่ได้รับประกันว่ามิติต้น ๆ จะเก็บความหมายไว้ได้ดี

In [19]:
sentences[0:3]

['แมวของฉันชอบนอนกลางแดด',
 'เจ้าเหมียวที่บ้านชอบไปนอนตากแดด',
 'ลูกแมวกำลังวิ่งไล่จับลูกบอล']

In [20]:
# สร้าง embedding เต็มเพียงครั้งเดียว
_, full_vectors = generate_dense_embeddings(sentences)
full_vectors = np.asarray(full_vectors, dtype=np.float64)
full_dimension = full_vectors.shape[1]

In [21]:
def l2_normalize(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, 1e-12)

def top_k_neighbors(similarity_matrix, k=3):
    # ไม่ให้แต่ละข้อความเลือกตัวเองเป็นเพื่อนบ้าน
    scores = similarity_matrix.copy()
    np.fill_diagonal(scores, -np.inf)
    return np.argsort(-scores, axis=1)[:, :k]

normalized_full = l2_normalize(full_vectors)
full_similarity = normalized_full @ normalized_full.T
full_top3 = top_k_neighbors(full_similarity, k=3)
pair_mask = np.triu(np.ones_like(full_similarity, dtype=bool), k=1)

In [22]:
dimensions = [d for d in [32, 64, 128, 256, 512, full_dimension] if d <= full_dimension]
dimensions = sorted(set(dimensions))
comparison_rows = []
similarity_by_dimension = {}

for dimension in dimensions:
    trimmed = l2_normalize(full_vectors[:, :dimension])  # ตัด prefix แล้ว normalize ใหม่
    trimmed_similarity = trimmed @ trimmed.T
    trimmed_top3 = top_k_neighbors(trimmed_similarity, k=3)
    similarity_by_dimension[dimension] = trimmed_similarity

    correlation = np.corrcoef(
        full_similarity[pair_mask], trimmed_similarity[pair_mask]
    )[0, 1]
    top3_overlap = np.mean([
        len(set(reference) & set(candidate)) / 3
        for reference, candidate in zip(full_top3, trimmed_top3)
    ])

    comparison_rows.append({
        "dimensions": dimension,
        "size_vs_full": dimension / full_dimension,
        "storage_saved": 1 - dimension / full_dimension,
        "similarity_correlation": correlation,
        "top_3_overlap": top3_overlap,
        "top_1_agreement": np.mean(trimmed_top3[:, 0] == full_top3[:, 0]),
    })

# วัดผล:
# - **Similarity correlation**: คะแนนความคล้ายของทุกคู่ยังสอดคล้องกับเวกเตอร์เต็มเพียงใด (ยิ่งใกล้ 1 ยิ่งดี)
# - **Top-3 overlap**: เพื่อนบ้าน 3 อันดับแรกยังเป็นชุดเดิมมากน้อยเพียงใด
# - **Top-1 agreement**: เพื่อนบ้านอันดับแรกยังเป็นข้อความเดิมหรือไม่

matryoshka_results = pd.DataFrame(comparison_rows)
display(matryoshka_results.style.format({
    "size_vs_full": "{:.1%}",
    "storage_saved": "{:.1%}",
    "similarity_correlation": "{:.3f}",
    "top_3_overlap": "{:.1%}",
    "top_1_agreement": "{:.1%}",
}))

,dimensions,size_vs_full,storage_saved,similarity_correlation,top_3_overlap,top_1_agreement
0,32,4.2%,95.8%,0.450,48.3%,20.0%
1,64,8.3%,91.7%,0.678,58.3%,60.0%
2,128,16.7%,83.3%,0.843,61.7%,75.0%
3,256,33.3%,66.7%,0.933,78.3%,80.0%
4,512,66.7%,33.3%,0.981,90.0%,95.0%
5,768,100.0%,0.0%,1.000,100.0%,100.0%


In [34]:
query_index = 1
example_dimensions = [d for d in [full_dimension, 256, 128, 64] if d in similarity_by_dimension]
retrieval_rows = []

print("Query:", sentences[query_index], "\n")
for dimension in example_dimensions:
    scores = similarity_by_dimension[dimension][query_index].copy()
    scores[query_index] = -np.inf
    print("Dimension:", dimension)
    
    for rank, document_index in enumerate(np.argsort(-scores)[:3], start=1):
        print(f"  * {sentences[document_index]} (RANK #{rank}, {scores[document_index]:.2f})")

    print()

Query: เจ้าเหมียวที่บ้านชอบไปนอนตากแดด 

Dimension: 768
  * แมวของฉันชอบนอนกลางแดด (RANK #1, 0.87)
  * แมวตัวนี้ชอบกินปลาทูน่า (RANK #2, 0.69)
  * เจ้าเหมียวรีบวิ่งหนีเมื่อได้ยินเสียงดัง (RANK #3, 0.67)

Dimension: 256
  * แมวของฉันชอบนอนกลางแดด (RANK #1, 0.87)
  * แมวตัวนี้ชอบกินปลาทูน่า (RANK #2, 0.71)
  * เจ้าเหมียวรีบวิ่งหนีเมื่อได้ยินเสียงดัง (RANK #3, 0.68)

Dimension: 128
  * แมวของฉันชอบนอนกลางแดด (RANK #1, 0.90)
  * แมวตัวนี้ชอบกินปลาทูน่า (RANK #2, 0.77)
  * แมวของฉันชอบเล่นกับกล่องกระดาษ (RANK #3, 0.77)

Dimension: 64
  * แมวของฉันชอบนอนกลางแดด (RANK #1, 0.89)
  * แมวของฉันชอบเล่นกับกล่องกระดาษ (RANK #2, 0.77)
  * เจ้าเหมียวรีบวิ่งหนีเมื่อได้ยินเสียงดัง (RANK #3, 0.75)



# Example 3: Simple Semantic Search

In [130]:
documents = [
    "แมวพันธุ์สฟิงซ์ (Sphynx) แทบไม่มีขนเลย จึงแทบไม่มีปัญหาขนร่วง เหมาะกับคนเป็นภูมิแพ้",
    "แมวพันธุ์เปอร์เซียมีขนยาวฟูสวยงาม แต่ต้องหวีขนทุกวันเพราะขนร่วงเยอะมาก",
    "อาหารแมวที่มีโปรตีนสูงช่วยให้แมวมีสุขภาพขนดีและแข็งแรง",
    "แมวเป็นสัตว์เลี้ยงลูกด้วยนมที่ได้รับความนิยมมากที่สุดในโลก",
]

In [132]:
embeddings = generate_dense_embeddings(documents)

In [133]:
def semantic_search(query, embeddings, top_k = 1):
    _, query_vectors = generate_dense_embeddings([query])
    query_vector = query_vectors[0]

    results = []
    documents, document_vectors = embeddings
    for document, document_vector in zip(documents, document_vectors):
        score = cosine_similarity(query_vector, document_vector)

        results.append({
            "document": document,
            "cosine_similarity": score,
        })

    results.sort(key=lambda result: result["cosine_similarity"],reverse=True,)
    return results[:top_k]

In [134]:
semantic_search("แมวพันธุ์ไหนขนไม่ร่วง", embeddings)

[{'document': 'แมวพันธุ์สฟิงซ์ (Sphynx) แทบไม่มีขนเลย จึงแทบไม่มีปัญหาขนร่วง เหมาะกับคนเป็นภูมิแพ้',
  'cosine_similarity': 0.7732186098831012}]

In [138]:
!uv pip install -q langchain-community faiss-cpu

In [141]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
import numpy as np



def semantic_search_faiss(query, embeddings, top_k = 1):
    
    documents, document_vectors = embeddings
    document_matrix = np.asarray(document_vectors, dtype=np.float32,)

    text_embeddings = list(zip(documents, document_vectors))

    # สร้าง FAISS Vector Store
    store = FAISS.from_embeddings(
        text_embeddings=text_embeddings,
        embedding=embedder,
        distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
    )

    # LangChain จะเรียก embedder.embed_query(query) โดยอัตโนมัติ
    matches = store.similarity_search_with_score(query=query, k=top_k,)

    results = []
    for document, score in matches:
        results.append({
            "document": document.page_content,
            "cosine_similarity": float(score),
        })

    return results

In [142]:
semantic_search_faiss("แมวพันธุ์ไหนขนไม่ร่วง", embeddings)

[{'document': 'แมวพันธุ์สฟิงซ์ (Sphynx) แทบไม่มีขนเลย จึงแทบไม่มีปัญหาขนร่วง เหมาะกับคนเป็นภูมิแพ้',
  'cosine_similarity': 0.7732189297676086}]